### Environment Setup & Helper Function

First, we set up the connection to the local Ollama server.

In [ ]:
# 1. Install Dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Pa

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# 2. Start Ollama Server in Background
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10) # Give the server time to spin up
print("Ollama server ready on http://localhost:11434")

Ollama server ready on http://localhost:11434


In [ ]:
# 3. Pull Models
!ollama pull llama3

In [ ]:
!ollama pull gemma

In [ ]:
# 4. Standard API Helper with Parameter Control
import requests
import json

OLLAMA_API_URL = "http://localhost:11434/api/generate"

def call_ollama(prompt, model="llama3", temperature=0.7, stream=False):
    """Helper function to interact with the local Ollama API."""
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream,
        "options": {
            "temperature": temperature
        }
    }

    resp = requests.post(OLLAMA_API_URL, json=payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Request failed: {resp.status_code}, {resp.text}")

    if stream:
        for line in resp.text.splitlines():
            if not line.strip(): continue
            obj = json.loads(line)
            print(obj.get("response", ""), end="")
        print("\n")
    else:
        return resp.json().get("response", "")

print("Environment Ready. Target API:", OLLAMA_API_URL)

Environment Ready. Target API: http://localhost:11434/api/generate


### Topic 1: Zero-Shot Prompting

**Concept:** Asking the model to perform a task without providing any prior examples.

**Practical Scenario:** A marketing team needs quick brainstorming for a new product launch.

In [ ]:
# The model relies entirely on its pre-trained weights to understand the task.
zero_shot_prompt = """
Write a catchy, 5-word advertising tagline for a new brand of smart, self-heating coffee mugs.
"""

In [ ]:
response = call_ollama(zero_shot_prompt, model="llama3", temperature=0.8)
print(f"Prompt:\n{zero_shot_prompt.strip()}")
print(f"\nResult:\n{response.strip()}\n")

Prompt:
Write a catchy, 5-word advertising tagline for a new brand of smart, self-heating coffee mugs.

Result:
"Warmth in Every Sip, Everywhere!"



### Topic 2: Few-Shot Prompting

**Concept:** Providing the model with a few examples (input-output pairs) to establish a clear pattern, tone, or format before asking it to process the actual target data.

**Practical Scenario:** Standardizing how product features are translated into customer-facing marketing benefits.

In [ ]:
few_shot_prompt = """
Task: Convert a technical product feature into a customer-centric marketing benefit.

Example 1:
Feature: 10,000mAh lithium-ion battery
Benefit: Power through your entire weekend without ever searching for an outlet.

Example 2:
Feature: IP68 water resistance rating
Benefit: Take your device anywhere—from dusty trails to unexpected downpours—with total peace of mind.

Example 3:
Feature: Active Noise Cancellation with 6 external microphones
Benefit: Immerse yourself in pure audio bliss and silence the chaos of your daily commute.

Target:
Feature: AI-powered dual-camera system with 1-inch sensor
Benefit:
"""

In [ ]:
response = call_ollama(few_shot_prompt, model="llama3", temperature=0.3)
print(f"Target Feature: AI-powered dual-camera system with 1-inch sensor")
print(f"Few-Shot Result:\n{response.strip()}\n")

Target Feature: AI-powered dual-camera system with 1-inch sensor
Few-Shot Result:
Here's a potential benefit:

Benefit: Capture life's most precious moments with stunning clarity, even in low-light conditions. With our AI-powered dual-camera system, you'll never miss a special moment or forget the details of your day.

This benefit focuses on the customer's emotional connection to their memories and experiences, rather than just listing technical features. It highlights the value that the product brings to the customer's life, making it more relatable and appealing.



### Topic 3: Role Prompting

**Concept:** Assigning a specific persona, profession, or identity to the LLM to drastically alter its vocabulary, tone, and perspective.

**Practical Scenario:** Writing promotional copy for the same product, but targeting two entirely different audiences (B2B vs. Gen-Z Consumers).

In [ ]:
product_details = "CloudSync Pro: A decentralized cloud storage router that encrypts data locally before backing it up to the blockchain."

In [ ]:
# Persona 1: Gen-Z Social Media Influencer
role_prompt_1 = f"""
You are a trendy, high-energy Gen-Z tech influencer on TikTok.
Write a short, engaging script (under 50 words) promoting this product to your followers. Use modern slang and emojis.

Product: {product_details}
"""

In [ ]:
print("Persona 1 (Gen-Z Influencer):")
print(call_ollama(role_prompt_1, model="llama3", temperature=0.8))

Persona 1 (Gen-Z Influencer):
"Yaaas, fam! 🤩 Just got my hands on the new #CloudSyncPro and I'm OBSESSED 💯! This decentralized storage game-changer encrypts your data LOCALLY 🔒 and backs it up to the BLOCKCHAIN 💸, keeping your files SAFE and SECURE 🔒. Trust me, you need this in your life 🤩 #BlockchainLife #DataPrivacy"


In [ ]:
# Persona 2: Enterprise IT Sales Executive
role_prompt_2 = f"""
You are a highly technical Enterprise IT Sales Executive.
Write a formal, concise LinkedIn post (under 50 words) promoting this product to Chief Information Security Officers (CISOs). Focus on security and compliance.

Product: {product_details}
"""

In [ ]:
print("\nPersona 2 (Enterprise IT Executive):")
print(call_ollama(role_prompt_2, model="llama3", temperature=0.4))


Persona 2 (Enterprise IT Executive):
"Attention CISOs! Ensure the confidentiality, integrity, and availability of your organization's sensitive data with CloudSync Pro. This innovative solution encrypts data locally and backs it up to the blockchain, providing unparalleled security and compliance for GDPR, HIPAA, and other regulations." #CloudSecurity #Compliance


### Topic 4: Prompt Templates

**Concept:** Using Python string formatting to dynamically generate prompts based on variables. This is the foundational step before using frameworks like LangChain.

**Practical Scenario:** Automating personalized email generation for a marketing CRM.

In [ ]:
def generate_marketing_email(customer_name, purchased_product, suggested_product, discount_code):
    """A reusable prompt template function for marketing CRM."""

    # The template relies on f-strings to inject variables dynamically
    prompt_template = f"""
    You are a friendly customer success manager for a tech gadget store.
    Write a short, personalized follow-up email to a customer.

    Customer Name: {customer_name}
    Recently Purchased: {purchased_product}
    Product to Cross-Sell: {suggested_product}
    Discount Offer: {discount_code}

    Instructions:
    1. Thank them for buying the {purchased_product}.
    2. Suggest the {suggested_product} as a perfect companion.
    3. Offer the discount code {discount_code} for 15% off their next purchase.
    4. Keep it under 100 words and professional.
    """

    return call_ollama(prompt_template, model="llama3", temperature=0.5)

In [ ]:
# Executing the template with dynamic data
customer_email = generate_marketing_email(
    customer_name="Alex",
    purchased_product="Wireless Mechanical Keyboard",
    suggested_product="Ergonomic Gel Wrist Rest",
    discount_code="COMFORT15"
)

print("Generated Template Email:\n")
print(customer_email)

Generated Template Email:

Subject: Welcome to the Wireless Mechanical Keyboard Family!

Dear Alex,

A huge thank you for choosing our Wireless Mechanical Keyboard! We're thrilled you've joined the family of satisfied customers.

As a proud owner, we think you'll love pairing your new keyboard with an Ergonomic Gel Wrist Rest. This amazing accessory will provide extra comfort and support during long typing sessions. And as a special offer, use COMFORT15 at checkout to receive 15% off your next purchase!

Looking forward to helping you stay comfortable and productive.

Best regards,
[Your Name]
Customer Success Manager


### Topic 5: Case Study — Marketing Content Generation Pipeline

**Concept:** Combining Prompt Templates, Role Prompting, and formatting instructions to create an automated end-to-end campaign generation pipeline.

**Practical Scenario:** Generating a cohesive marketing suite (Email, Social Post, and Ad Copy) for a single product launch in one run.

In [ ]:
def run_marketing_campaign(product_name, target_audience, core_benefit):
    """Generates a multi-channel marketing campaign."""

    campaign_prompt = f"""
    You are the Chief Marketing Officer for a top-tier advertising agency.
    We are launching a new product.

    Product Context:
    - Product Name: {product_name}
    - Target Audience: {target_audience}
    - Core Benefit: {core_benefit}

    Task:
    Generate a cohesive marketing campaign consisting of three parts.
    Format your response strictly using Markdown headers for each section.

    # 1. Subject Line
    Write one high-converting email subject line.

    # 2. Instagram Caption
    Write a visually appealing Instagram caption with exactly 3 relevant hashtags.

    # 3. Google Search Ad
    Write a Google Ad headline (max 30 chars) and description (max 90 chars).
    """

    # Using stream=True for a better UX during live training demonstrations
    print(f"Generating campaign for: {product_name}...\n")
    call_ollama(campaign_prompt, model="llama3", temperature=0.7, stream=True)

In [ ]:
# Run the Case Study
run_marketing_campaign(
    product_name="EcoBreeze Smart Fan",
    target_audience="Environmentally conscious homeowners looking to reduce AC bills",
    core_benefit="Uses AI to detect room temperature and automatically adjusts flow to save up to 40% on energy costs."
)

Generating campaign for: EcoBreeze Smart Fan...

Here is the cohesive marketing campaign:

# 1. Subject Line
"Beat the Heat, Not Your Budget: Introducing EcoBreeze Smart Fan"

This subject line aims to capture the attention of environmentally conscious homeowners by highlighting the product's ability to save them money on energy costs.

# 2. Instagram Caption
"Stay cool, stay green! Meet EcoBreeze Smart Fan, the AI-powered fan that adjusts airflow to reduce your AC bills by up to 40%! Say goodbye to high energy costs and hello to a more sustainable lifestyle #EcoFriendly #SmartHome #EnergyEfficiency"

This caption uses visually appealing language and hashtags to grab the attention of Instagram users who are interested in environmentally friendly products.

# 3. Google Search Ad
**Headline:** "Save Up to 40% on AC Bills with EcoBreeze"
**Description:** "Introducing EcoBreeze Smart Fan, an AI-powered fan that adjusts airflow for maximum energy efficiency. Reduce your AC bills and help th